# PCBUILD QLoRA v18: tiếp tục Qwen3.5-4B với micro-batch 2

V17 đã train Qwen3.5-4B Instruct từ model nền trên T4, 6 epochs, effective batch 4, best validation loss 0.9822. V18 tiếp tục từ adapter tốt nhất v17 với micro-batch 2 × accumulation 2 (effective batch 4), learning rate 3e-6, tối đa 6 epochs và early stopping. Thử nghiệm này tăng batch xử lý đồng thời để tận dụng thêm VRAM/throughput trên T4.

Nếu batch 2 gây CUDA OOM, v17 vẫn giữ nguyên trên Drive. Checkpoint của v18 được lưu riêng và có thể resume. Không nạp adapter 2B vào model 4B.

In [ ]:
import hashlib
import json
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path
from google.colab import drive, files

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/PCBUILD")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
BUNDLE_PATH = DRIVE_ROOT / "pcbuild-colab-bundle.zip"

LOCAL_BUNDLE = Path("/content/pcbuild-colab-bundle-v15.zip")
if not BUNDLE_PATH.exists() or hashlib.sha256(BUNDLE_PATH.read_bytes()).hexdigest() != "17ce86994806073558a7156b2986b4164a040676158c3a8cfbabe97c31bb6958":
    if not LOCAL_BUNDLE.is_file():
        raise FileNotFoundError(f"Upload the new bundle with the VS Code Colab Explorer: {LOCAL_BUNDLE}")
    bundle_bytes = LOCAL_BUNDLE.read_bytes()
    if hashlib.sha256(bundle_bytes).hexdigest() != "17ce86994806073558a7156b2986b4164a040676158c3a8cfbabe97c31bb6958":
        raise ValueError("Uploaded Colab bundle does not match the expected SHA-256.")
    BUNDLE_PATH.write_bytes(bundle_bytes)
    print("Verified the v15 bundle and copied it to Google Drive.")

with zipfile.ZipFile(BUNDLE_PATH) as archive:
    manifest = json.loads(archive.read("COLAB_BUNDLE_MANIFEST.json"))
    for name, expected_hash in manifest["files"].items():
        content = archive.read(name)
        if hashlib.sha256(content).hexdigest() != expected_hash:
            raise ValueError(f"Bundle checksum mismatch: {name}")
    for member in archive.namelist():
        path = Path(member)
        if path.is_absolute() or ".." in path.parts:
            raise ValueError(f"Unsafe archive path: {member}")
    PROJECT = Path("/content/pcbuild")
    if PROJECT.exists():
        shutil.rmtree(PROJECT)
    archive.extractall(PROJECT, members=[item for item in archive.infolist() if item.filename != "COLAB_BUNDLE_MANIFEST.json"])

TRAINING = PROJECT / "training"
print(f"Verified {len(manifest['files'])} files; extracted source to {PROJECT}")
print(f"Persistent Drive directory: {DRIVE_ROOT}")

## Cài môi trường và xác nhận GPU

Colab cần bật GPU tại Runtime → Change runtime type. Cell này giữ nguyên PyTorch/CUDA mà Colab cung cấp, chỉ cài các thư viện fine-tune đã được pin. Nếu GPU không hỗ trợ bfloat16, script tự chuyển sang float16.

In [ ]:
packages = [
    "transformers==5.20.2",
    "peft==0.21.2",
    "accelerate==1.15.0",
    "safetensors==0.8.0",
    "bitsandbytes==0.50.2",
    "Pillow==12.3.0",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU. Enable a GPU runtime in Google Colab and rerun.")
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
capability = torch.cuda.get_device_capability(0)
print("native bf16:", torch.cuda.is_bf16_supported() and capability[0] >= 8)
print("HF auth token: not required for the public base model")

## Dataset và cấu hình v18
Dùng cùng dữ liệu và split của v17 để kiểm tra micro-batch 2. Khởi tạo từ adapter v17 tốt nhất, giảm learning rate để refinement. Cùng split chỉ giúp so sánh run, không xác nhận tổng quát hóa.

In [ ]:
subprocess.check_call([
    sys.executable,
    str(TRAINING / "prepare_colab_dataset.py"),
    "--output",
    str(TRAINING / "data" / "pc_colab_sft.jsonl"),
])
DATA_FILE = TRAINING / "data" / "pc_colab_sft.jsonl"
BASELINE_RUN_NAME = "pcbuild-qwen3.5-2b-qlora-v14-colab"
BASELINE_ADAPTER = DRIVE_ROOT / "runs" / BASELINE_RUN_NAME / "checkpoints" / "final"
PARENT_RUN_NAME = "pcbuild-qwen3.5-4b-qlora-v17-colab"
PARENT_RUN_DRIVE = DRIVE_ROOT / "runs" / PARENT_RUN_NAME
SEED_ADAPTER = PARENT_RUN_DRIVE / "checkpoints" / "final"
RUN_NAME = "pcbuild-qwen3.5-4b-qlora-v18-mb2-colab"
MODEL_ID = "Qwen/Qwen3.5-4B"
RUN_DRIVE = DRIVE_ROOT / "runs" / RUN_NAME
RUN_DRIVE.mkdir(parents=True, exist_ok=True)
if not (SEED_ADAPTER / "adapter_model.safetensors").is_file():
    raise FileNotFoundError(f"Parent v17 adapter missing: {SEED_ADAPTER}")
if not (BASELINE_ADAPTER / "training_metrics.json").is_file():
    raise FileNotFoundError(f"v14 comparison metrics missing: {BASELINE_ADAPTER}")


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_tree(root):
    digest = hashlib.sha256()
    for path in sorted(item for item in root.rglob("*") if item.is_file()):
        digest.update(path.relative_to(root).as_posix().encode("utf-8"))
        digest.update(bytes.fromhex(sha256_file(path)))
    return digest.hexdigest()

PARENT_METRICS = json.loads((SEED_ADAPTER / "training_metrics.json").read_text(encoding="utf-8"))
BASELINE_METRICS = json.loads((BASELINE_ADAPTER / "training_metrics.json").read_text(encoding="utf-8"))
RUN_CONFIG = {
    "baseline_run": BASELINE_RUN_NAME,
    "baseline_validation_loss": BASELINE_METRICS["eval_loss"],
    "parent_run": PARENT_RUN_NAME,
    "parent_validation_loss": PARENT_METRICS["eval_loss"],
    "model": MODEL_ID,
    "init_adapter": str(SEED_ADAPTER),
    "seed_adapter_sha256": sha256_tree(SEED_ADAPTER),
    "dataset_sha256": sha256_file(DATA_FILE),
    "training_script_sha256": sha256_file(TRAINING / "train_qlora.py"),
    "dataset_builder_sha256": sha256_file(TRAINING / "prepare_colab_dataset.py"),
    "dataset_examples": sum(1 for line in DATA_FILE.open(encoding="utf-8") if line.strip()),
    "epochs": 6,
    "learning_rate": 3e-6,
    "micro_batch_size": 2,
    "effective_batch_size": 4,
    "gradient_accumulation_steps": 2,
    "max_length": 768,
    "seed": 42,
    "save_eval_steps": 8,
    "early_stopping_patience": 3,
}
manifest_path = RUN_DRIVE / "run_config.json"
if manifest_path.exists():
    existing = json.loads(manifest_path.read_text(encoding="utf-8"))
    if existing != RUN_CONFIG:
        raise ValueError("Run config/data changed. Use a new RUN_NAME to avoid resuming an incompatible checkpoint.")
else:
    manifest_path.write_text(json.dumps(RUN_CONFIG, indent=2), encoding="utf-8")
print(json.dumps(RUN_CONFIG, indent=2))

## Validation seed adapter v14
Metric của v14 trên cùng split là mốc chọn checkpoint tốt nhất hiện tại.


In [ ]:
PARENT_METRICS = json.loads((SEED_ADAPTER / "training_metrics.json").read_text(encoding="utf-8"))
print(f"v14 validation loss: {PARENT_METRICS['eval_loss']}")

## Train v18: micro-batch 2
Micro-batch 2 × accumulation 2 giữ effective batch 4. Run khởi tạo từ adapter v17, lưu checkpoint riêng lên Drive và dừng theo validation. Nếu OOM, giữ v17 làm candidate.

In [ ]:
LOCAL_OUTPUT = TRAINING / "output" / RUN_NAME
CHECKPOINT_DRIVE = RUN_DRIVE / "checkpoints"
LOCAL_OUTPUT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DRIVE.mkdir(parents=True, exist_ok=True)
FINAL_ADAPTER = CHECKPOINT_DRIVE / "final"
FINAL_METRICS = FINAL_ADAPTER / "training_metrics.json"

if FINAL_METRICS.is_file():
    final_metrics = json.loads(FINAL_METRICS.read_text(encoding="utf-8"))
    print("Run already completed; keeping the saved best adapter.")
    print("Best validation loss:", final_metrics.get("eval_loss"))
    print("Adapter:", FINAL_ADAPTER)
else:
    resume_path = None
    latest_file = CHECKPOINT_DRIVE / "latest_checkpoint.json"
    if latest_file.exists():
        latest = json.loads(latest_file.read_text(encoding="utf-8"))
        checkpoint_name = latest["checkpoint"]
        if Path(checkpoint_name).name != checkpoint_name or not checkpoint_name.startswith("checkpoint-"):
            raise ValueError(f"Unsafe checkpoint name in Drive manifest: {checkpoint_name}")
        drive_checkpoint = CHECKPOINT_DRIVE / checkpoint_name
        state_file = drive_checkpoint / "trainer_state.json"
        if not state_file.is_file():
            raise FileNotFoundError(f"Checkpoint is incomplete: {state_file}")
        state = json.loads(state_file.read_text(encoding="utf-8"))
        if int(state["global_step"]) != int(latest["global_step"]):
            raise ValueError("Drive checkpoint state does not match latest_checkpoint.json")
        local_checkpoint = LOCAL_OUTPUT / checkpoint_name
        if local_checkpoint.exists():
            shutil.rmtree(local_checkpoint)
        shutil.copytree(drive_checkpoint, local_checkpoint)
        resume_path = local_checkpoint
        print(f"Resuming from Drive checkpoint {checkpoint_name} at step {latest['global_step']}")
    else:
        print("No complete checkpoint; continuing from the best v17 adapter with micro-batch 2.")

    command = [
        sys.executable, str(TRAINING / "train_qlora.py"),
        "--epochs", str(RUN_CONFIG["epochs"]),
        "--learning-rate", str(RUN_CONFIG["learning_rate"]),
        "--model", RUN_CONFIG["model"],
        "--data", str(DATA_FILE),
        "--output", str(LOCAL_OUTPUT),
        "--save-steps", str(RUN_CONFIG["save_eval_steps"]),
        "--eval-steps", str(RUN_CONFIG["save_eval_steps"]),
        "--logging-steps", "1",
        "--micro-batch-size", str(RUN_CONFIG["micro_batch_size"]),
        "--effective-batch-size", str(RUN_CONFIG["effective_batch_size"]),
        "--early-stopping-patience", str(RUN_CONFIG["early_stopping_patience"]),
        "--checkpoint-backup-dir", str(CHECKPOINT_DRIVE),
    ]
    if RUN_CONFIG["init_adapter"]:
        command.extend(["--init-adapter", str(SEED_ADAPTER)])
    if resume_path:
        command.extend(["--resume-from-checkpoint", str(resume_path)])

    training_log = RUN_DRIVE / "training.log"
    env = dict(**__import__("os").environ)
    env["WANDB_DISABLED"] = "true"
    started_at = time.time()
    with training_log.open("a", encoding="utf-8") as log_stream:
        process = subprocess.Popen(command, cwd=PROJECT, stdout=log_stream, stderr=subprocess.STDOUT, env=env)
        while process.poll() is None:
            try:
                process.wait(timeout=45)
            except subprocess.TimeoutExpired:
                heartbeat_path = CHECKPOINT_DRIVE / "heartbeat.json"
                if heartbeat_path.exists() and heartbeat_path.stat().st_mtime >= started_at:
                    heartbeat = json.loads(heartbeat_path.read_text(encoding="utf-8"))
                    print("Progress:", json.dumps(heartbeat, ensure_ascii=False))
                else:
                    print("Training is loading the model; log:", training_log)

    exit_code = process.returncode
    tail = training_log.read_text(encoding="utf-8", errors="replace").splitlines()[-60:]
    print("\n".join(tail))
    if exit_code:
        raise RuntimeError(f"Training stopped with exit code {exit_code}. Rerun this cell to resume from the latest saved checkpoint.")
    if not FINAL_METRICS.is_file():
        raise FileNotFoundError(f"Training ended without final metrics: {FINAL_METRICS}")
    final_metrics = json.loads(FINAL_METRICS.read_text(encoding="utf-8"))
    print("Best validation loss:", final_metrics.get("eval_loss"))
    print("Final adapter and manifest:", FINAL_ADAPTER)

## So sánh v18 với v17 và v14
Chọn checkpoint v18 theo validation loss; so sánh cả với v17 (4B) và v14 (2B). Đây là cùng split đã dùng trước đó, chưa phải held-out evaluation độc lập.

In [ ]:
FINAL_ADAPTER = CHECKPOINT_DRIVE / "final"
V18_METRICS = json.loads((FINAL_ADAPTER / "training_metrics.json").read_text(encoding="utf-8"))
selected_loss = V18_METRICS["eval_loss"]
print(f"v14 2B validation loss: {RUN_CONFIG['baseline_validation_loss']:.4f}")
print(f"v17 4B validation loss: {RUN_CONFIG['parent_validation_loss']:.4f}")
print(f"v18 4B best validation loss: {selected_loss:.4f}")
print("Validation improved vs v17:", selected_loss < RUN_CONFIG["parent_validation_loss"])
print("Adapter:", FINAL_ADAPTER)
print("Same-split comparison only; held-out behavioral test was not used.")